# 🗄️ NOVA — GRPO Training Notebook

**Train Qwen2.5-1.5B to be a Senior DBA Agent using GRPO + LoRA**

Each episode: reset DB → agent picks index strategy → reward = did query cost drop?

> **Runtime:** GPU (T4 or better). `Runtime → Change runtime type → T4 GPU`  
> **Time:** ~20–40 minutes for 30 episodes

## 1. Clone Repository

In [ ]:
!git clone https://github.com/itsflash44/db-tune-project.git
%cd db-tune-project

## 2. Install Dependencies

In [ ]:
!pip install -q openenv-core>=0.1.13 trl>=0.8.0 peft>=0.10.0 \
    accelerate>=0.28.0 transformers>=4.40.0 datasets>=2.18.0 \
    fastapi 'uvicorn[standard]' openai websockets

## 3. Configuration

In [ ]:
import os

HF_TOKEN     = ''    # paste your HF token here
HF_REPO      = ''    # optional: 'yourname/nova-dba-lora'
NUM_EPISODES = 30

os.environ['HF_TOKEN']     = HF_TOKEN
os.environ['HF_REPO']      = HF_REPO
os.environ['NUM_EPISODES'] = str(NUM_EPISODES)
print(f'Episodes: {NUM_EPISODES}')

## 4. Start Local WebSocket Server

In [ ]:
import subprocess, time, sys, os, requests

# Start the FastAPI server (now has /ws WebSocket endpoint)
server_proc = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'server.app:app',
     '--host', '0.0.0.0', '--port', '7860', '--log-level', 'warning'],
    cwd='/content/db-tune-project',
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Wait for server to be ready
ENV_URL = 'http://localhost:7860'
os.environ['ENV_BASE_URL'] = ENV_URL

for i in range(15):  # wait up to 15 seconds
    time.sleep(1)
    try:
        r = requests.get(f'{ENV_URL}/', timeout=2)
        if r.status_code == 200:
            print(f'✅ Server ready: {r.json()}')
            break
    except Exception:
        print(f'  Waiting... ({i+1}s)', end='\r')
else:
    # Print stderr if server didn't start
    stdout, stderr = server_proc.communicate(timeout=2)
    print('❌ Server failed to start:')
    print(stderr.decode()[:2000])

## 5. Verify WebSocket Connectivity

In [ ]:
import sys
sys.path.insert(0, '/content/db-tune-project')

from client import DBEnvClient, DBAction

print(f'Connecting via WebSocket to {ENV_URL}...')
env = DBEnvClient(base_url=ENV_URL)

with env.sync() as sync_env:
    result = sync_env.reset(task='easy')
    obs = result.observation
    print(f'✅ Connected!')
    print(f'   Initial cost   : {obs.query_cost}')
    print(f'   Storage budget : {obs.storage_budget}')
    print(f'   Indices        : {obs.current_indices}')

print('\n🚀 Environment ready for training.')

## 6. Import Training Utilities

In [ ]:
import logging
from reward_functions import reward_total, StepState
from train import make_rollout_func, SYSTEM_PROMPT, OUTPUT_DIR

logging.basicConfig(level=logging.INFO)
print('System prompt (first 200 chars):')
print(SYSTEM_PROMPT[:200])
print(f'\n✅ Training utilities imported')

## 7. GRPO Training Setup

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer

MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

dataset = Dataset.from_dict({'prompt': ['Diagnose and optimise the database.'] * NUM_EPISODES})

grpo_config = GRPOConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    logging_steps=5,
    save_steps=20,
    temperature=0.3,
    max_completion_length=256,
    report_to='none',
)

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    bias='none', task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
)

trainer = GRPOTrainer(
    model=MODEL_ID,
    processing_class=tokenizer,
    reward_funcs=[lambda prompts, completions, **kw: [1.0] * len(prompts)],
    args=grpo_config,
    train_dataset=dataset,
    rollout_func=make_rollout_func(env, tokenizer, ['easy', 'medium', 'hard']),
    peft_config=lora_config,
)
print('✅ GRPOTrainer initialized')

## 8. Train!

In [ ]:
print(f'Starting GRPO training — {NUM_EPISODES} episodes...')
trainer.train()
trainer.save_model(str(OUTPUT_DIR))
print(f'\n✅ Model saved to {OUTPUT_DIR}')

## 9. Reward Curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

log_path = next(Path('outputs').glob('**/reward_log.csv'), None)
if log_path:
    df = pd.read_csv(log_path)
    df['rolling_reward'] = df['total_reward'].rolling(5, min_periods=1).mean()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.patch.set_facecolor('#0d1117')
    fig.suptitle('NOVA DBA Agent — Training Reward Curve', fontsize=14, fontweight='bold', color='white')

    for task, grp in df.groupby('task'):
        ax1.plot(grp['episode'], grp['total_reward'], alpha=0.4, linewidth=1, label=f'{task}')
    ax1.plot(df['episode'], df['rolling_reward'], color='white', linewidth=2.5, label='Rolling avg (5)')
    ax1.axhline(y=1.5, color='lime', linestyle='--', linewidth=1.5, label='Target (+1.5)')
    ax1.set_facecolor('#161b22'); ax1.tick_params(colors='white')
    ax1.set_title('Total Reward per Episode', color='white')
    ax1.set_xlabel('Episode', color='white'); ax1.set_ylabel('Reward', color='white')
    ax1.legend(facecolor='#21262d', labelcolor='white', fontsize=8)

    for task, grp in df.groupby('task'):
        ax2.plot(grp['episode'], grp['final_cost'], linewidth=2, label=task)
    ax2.axhline(y=10, color='lime', linestyle='--', linewidth=1.5, label='Target (cost=10)')
    ax2.set_facecolor('#161b22'); ax2.tick_params(colors='white')
    ax2.set_title('Final Query Cost per Episode', color='white')
    ax2.set_xlabel('Episode', color='white'); ax2.set_ylabel('Cost', color='white')
    ax2.legend(facecolor='#21262d', labelcolor='white', fontsize=8)

    plt.tight_layout()
    out = OUTPUT_DIR / 'reward_curve.png'
    plt.savefig(str(out), dpi=150, bbox_inches='tight', facecolor='#0d1117')
    plt.show()
    print(f'✅ Saved — download from Files sidebar → outputs/ folder')
else:
    print('No reward log yet — run training first.')